In [3]:
import os 
import openai
from dotenv import load_dotenv

In [5]:
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

In [13]:
from typing import List
from pydantic import BaseModel, Field
from langchain_core.utils.function_calling import convert_pydantic_to_openai_function

In [14]:
class Tagging(BaseModel):
    """Tag the piece of text with particular information."""
    sentiment : str = Field(description="The sentiment of the text.")
    language : str = Field(description="The language of the text. (should be ISO 639-1 code)")

In [18]:
from langchain.prompts import ChatPromptTemplate
from langchain_openai.chat_models import ChatOpenAI

In [19]:
model = ChatOpenAI(model="gpt-4o", temperature=0)

In [20]:
tagging_functions = [convert_pydantic_to_openai_function(Tagging)]

In [21]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Think carefully, and then tag the text as instructed"),
    ("user", "{input}")
])

In [22]:
model_with_functions = model.bind(
    functions=tagging_functions,
    function_call={"name": "Tagging"},
)

In [23]:
tagging_chain = prompt | model_with_functions

In [24]:
tagging_chain.invoke({"input": "te quiero mucho, pero no puedo estar contigo."})

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"sentiment":"negative","language":"es"}', 'name': 'Tagging'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 104, 'total_tokens': 115, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-11-20', 'system_fingerprint': 'fp_ee1d74bde0', 'id': 'chatcmpl-BZzN2R82gkj2DQ4J8gVACdl7dFuWM', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--f59e82f9-33fe-4e8e-913a-7d47572f44b8-0', usage_metadata={'input_tokens': 104, 'output_tokens': 11, 'total_tokens': 115, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [25]:
from langchain.output_parsers.openai_functions import JsonOutputFunctionsParser

In [26]:
tagging_chain = prompt | model_with_functions | JsonOutputFunctionsParser()

In [27]:
tagging_chain.invoke({"input": "non mi piace questo cibo"})

{'sentiment': 'negative', 'language': 'it'}

### Extraction

In [28]:
from typing import Optional
class Person(BaseModel):
    """Information about a person."""
    name: str = Field(description="The name of the person.")
    age: Optional[int] = Field(description="The age of the person.")

In [29]:
class Information(BaseModel):
    """Information to extract."""
    people: List[Person] = Field(description="A list of info about people.")

In [30]:
convert_pydantic_to_openai_function(Information)

{'name': 'Information',
 'description': 'Information to extract.',
 'parameters': {'properties': {'people': {'description': 'A list of info about people.',
    'items': {'description': 'Information about a person.',
     'properties': {'name': {'description': 'The name of the person.',
       'type': 'string'},
      'age': {'anyOf': [{'type': 'integer'}, {'type': 'null'}],
       'description': 'The age of the person.'}},
     'required': ['name', 'age'],
     'type': 'object'},
    'type': 'array'}},
  'required': ['people'],
  'type': 'object'}}

In [31]:
extraction_functions = [convert_pydantic_to_openai_function(Information)]
extraction_model = model.bind(functions=extraction_functions, function_call={"name": "Information"})

In [34]:
extraction_model.invoke("I met John, who is 25 years old, and Mary, who is 30 years old and their sister Anna")

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"people":[{"name":"John","age":25},{"name":"Mary","age":30},{"name":"Anna","age":null}]}', 'name': 'Information'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 111, 'total_tokens': 141, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-11-20', 'system_fingerprint': 'fp_ee1d74bde0', 'id': 'chatcmpl-BZzdy10BsegP6NnPJ7HkRlghgitn6', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--f06ea258-a642-4430-8ea6-2d43e4171942-0', usage_metadata={'input_tokens': 111, 'output_tokens': 30, 'total_tokens': 141, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [35]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract the information as instructed, if not explicitly provided do not guess. Extract partial info."),
    ("user", "{input}")
])

In [36]:
extraction_chain = prompt | extraction_model 

In [37]:
extraction_chain.invoke("I met John, who is 25 years old, and Mary, who is 30 years old and their sister Anna")

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"people":[{"name":"John","age":25},{"name":"Mary","age":30},{"name":"Anna","age":null}]}', 'name': 'Information'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 129, 'total_tokens': 159, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-11-20', 'system_fingerprint': 'fp_ee1d74bde0', 'id': 'chatcmpl-BZziBZc3eQlY4ma4FWlN7YDkPQljt', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--269b3d1a-ccb1-456a-8d86-b71301ee27a2-0', usage_metadata={'input_tokens': 129, 'output_tokens': 30, 'total_tokens': 159, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [38]:
extraction_chain = prompt | extraction_model | JsonOutputFunctionsParser()

In [ ]:
extraction_chain.invoke("I met John, who is 25 years old, and Mary, who is 30 years old and their sister Anna")

{'people': [{'name': 'John', 'age': 25},
  {'name': 'Mary', 'age': 30},
  {'name': 'Anna', 'age': None}]}

In [40]:
from langchain.output_parsers.openai_functions import JsonKeyOutputFunctionsParser

In [41]:
extraction_chain = prompt | extraction_model | JsonKeyOutputFunctionsParser(key_name="people")

In [43]:
extraction_chain.invoke("I met John, who is 25 years old, and Mary, who is 30 years old and their sister Anna")

[{'name': 'John', 'age': 25},
 {'name': 'Mary', 'age': 30},
 {'name': 'Anna', 'age': None}]